# 02 - Train and evaluate

Run `01_Baseline.ipynb` first, then restart the kernel to free GPU memory.
Train a LoRA adapter on 72 sequences and keep the epoch with the lowest validation loss.
Compare it with the baseline on the same eight validation sequences.

In [1]:
import json
import os
import random
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
from peft import LoraConfig
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

# Find the project folder when running from the root or notebooks folder.
ROOT = Path.cwd().resolve()
if not (ROOT / "Scripts").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "Scripts" / "trainer.py").is_file():
    raise FileNotFoundError("Run this notebook from the project or notebooks folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from Scripts.dataUtils import (
    DEFAULT_TASK_PROMPT,
    MangaCollator,
    MangaPromptBuilder,
    MangaSequenceDataset,
    load_labels,
)
from Scripts.evaluator import (
    calculate_metrics,
    display_metric_summary,
    evaluate_model,
    inspect_prediction,
    load_prediction_records,
)
from Scripts.trainer import build_lora_model, build_trainer, build_training_args

SEED = 42
MODEL_PATH = Path(os.environ.get("MANGA_VLM_MODEL_PATH", r"C:\models\HF\Qwen2.5-VL-3B-Instruct"))
LABELS_PATH = ROOT / "TaskData" / "development" / "labels.jsonl"
IMAGE_ROOT = ROOT / "TaskData" / "development" / "images"
SPLIT_PATH = ROOT / "outputs" / "canonical_split_seed42.json"
# Match OUTPUT_DIR in 01_Baseline.ipynb.
BASELINE_DIR = ROOT / "outputs" / "canonical_base_baseline_better_prompt"
RUN_DIR = ROOT / "artifacts" / "canonical_lora"
EVAL_DIR = ROOT / "outputs" / "canonical_lora_eval"
MAX_NEW_TOKENS = 1024
MINIMUM_SIMILARITY = 0.50
REPETITION_PENALTY = 1.05

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

W1001 17:12:17.214000 21740 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


In [2]:
if not SPLIT_PATH.is_file():
    raise FileNotFoundError("Run 01_Baseline.ipynb first to save the split")
if not (BASELINE_DIR / "predictions.jsonl").is_file():
    raise FileNotFoundError(f"Run the baseline evaluation first: {BASELINE_DIR}")

split = json.loads(SPLIT_PATH.read_text(encoding="utf-8"))
labels = load_labels(LABELS_PATH)
train_ids = split["train_ids"]
validation_ids = split["validation_ids"]
assert set(train_ids).isdisjoint(validation_ids)
assert set(train_ids) | set(validation_ids) == set(labels)

train_dataset = MangaSequenceDataset(LABELS_PATH, IMAGE_ROOT, train_ids)
validation_dataset = MangaSequenceDataset(LABELS_PATH, IMAGE_ROOT, validation_ids)
print(f"Training: {len(train_dataset)}, validation: {len(validation_dataset)}")

Training: 72, validation: 8


In [3]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for 4-bit LoRA training")
if not MODEL_PATH.is_dir():
    raise FileNotFoundError(f"Local model not found: {MODEL_PATH}")

processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    min_pixels=128 * 28 * 28,
    max_pixels=256 * 28 * 28,
)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    device_map={"": 0},
)

prompt_builder = MangaPromptBuilder(DEFAULT_TASK_PROMPT)
train_collator = MangaCollator(
    processor, process_vision_info, prompt_builder, train_mode=True,
)
eval_collator = MangaCollator(
    processor, process_vision_info, prompt_builder,
    train_mode=False, return_metadata=True,
)

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

In [4]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = build_lora_model(base_model, lora_config)
model.print_trainable_parameters()

# The helper uses batch size 1, gradient accumulation 8, and learning rate 1e-4.
# It evaluates and saves each epoch, then loads the model with the lowest loss.
training_args = build_training_args(
    output_dir=str(RUN_DIR),
    num_train_epochs=5,
    logging_steps=5,
    warmup_steps=5,
)
trainer = build_trainer(
    model, train_dataset, train_collator, training_args, validation_dataset,
)

trainable params: 37,152,768 || all params: 3,791,775,744 || trainable%: 0.9798


In [5]:
train_result = trainer.train()
display(pd.Series(train_result.metrics, name="training"))

FINAL_DIR = RUN_DIR / "final"
trainer.save_model(str(FINAL_DIR))
processor.save_pretrained(str(FINAL_DIR))
print(f"Saved best adapter to {FINAL_DIR}")

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
1,0.614167,0.641589
2,0.528336,0.575704
3,0.390040,0.555661
4,0.355099,0.543526
5,0.355144,0.542737


train_runtime               3.910640e+03
train_samples_per_second    9.200000e-02
train_steps_per_second      1.200000e-02
total_flos                  1.441683e+16
train_loss                  4.462395e-01
epoch                       5.000000e+00
Name: training, dtype: float64

Saved best adapter to C:\Users\ESHAAN\BYTE\Applied-Task\manga_vlm_crashcourse\artifacts\canonical_lora\final


In [6]:
tuned_metrics = evaluate_model(
    model=model,
    processor=processor,
    dataset=validation_dataset,
    collator=eval_collator,
    batch_size=1,
    max_new_tokens=MAX_NEW_TOKENS,
    minimum_similarity=MINIMUM_SIMILARITY,
    repetition_penalty=REPETITION_PENALTY,
    output_dir=EVAL_DIR,
)
display_metric_summary(tuned_metrics)

Evaluated 1/8 sequences
Evaluated 2/8 sequences
Evaluated 3/8 sequences
Evaluated 4/8 sequences
Evaluated 5/8 sequences
Evaluated 6/8 sequences
Evaluated 7/8 sequences
Evaluated 8/8 sequences


,Result,Value,What it tells us
0,Format valid?,0.625,5/8 outputs usable; check page-count diagnostics
1,Text error rate (lower is better),0.944,2378 reference characters on valid outputs
2,Dialogue F1,0.468,"33 matched, 40 missing, 35 extra; 66 target li..."
3,Speaker accuracy,0.606,20/33 matched dialogues correct after characte...


In [7]:
baseline_records = load_prediction_records(BASELINE_DIR / "predictions.jsonl")
assert sorted(record["sequence_id"] for record in baseline_records) == sorted(validation_ids)
baseline_metrics = calculate_metrics(
    baseline_records, minimum_similarity=MINIMUM_SIMILARITY,
)
metric_names = [
    "cer_valid_outputs",
    "cer_including_failures_as_deletions",
    "cer_substitution_rate",
    "cer_deletion_rate",
    "cer_insertion_rate",
    "dialogue_text_match_f1",
    "matched_speaker_accuracy",
    "b3_f1",
    "identity_switch_rate",
    "narration_f1",
    "json_parse_rate",
    "schema_compliance_rate",
    "sequence_exact_match",
]
comparison = pd.DataFrame({
    "base_model": [baseline_metrics[name] for name in metric_names],
    "tuned": [tuned_metrics[name] for name in metric_names],
}, index=metric_names)
comparison["delta"] = comparison["tuned"] - comparison["base_model"]
display(comparison.round(4))

,base_model,tuned,delta
cer_valid_outputs,1.2186,0.9437,-0.2749
cer_including_failures_as_deletions,1.0528,0.9752,-0.0776
cer_substitution_rate,0.0077,0.0627,0.0550
cer_deletion_rate,0.7922,0.5248,-0.2674
cer_insertion_rate,0.4187,0.3562,-0.0625
dialogue_text_match_f1,0.2535,0.4681,0.2146
matched_speaker_accuracy,0.1111,0.6061,0.4949
b3_f1,1.0000,0.6264,-0.3736
identity_switch_rate,0.5000,0.3333,-0.1667
narration_f1,0.0000,0.0000,0.0000


In [8]:
records = load_prediction_records(EVAL_DIR / "predictions.jsonl")
rows = []
for record in records:
    metrics = calculate_metrics([record], minimum_similarity=MINIMUM_SIMILARITY)
    rows.append({
        "sequence_id": record["sequence_id"],
        "format_valid": bool(metrics["schema_compliance_rate"]),
        "text_error_rate": metrics["cer_valid_outputs"],
        "failure_penalty_cer": metrics["cer_including_failures_as_deletions"],
        "speaker_accuracy": metrics["matched_speaker_accuracy"],
        "dialogue_f1": metrics["dialogue_text_match_f1"],
        "generated_tokens": record["generated_tokens"],
        "hit_max_new_tokens": record["hit_max_new_tokens"],
        "repetition_loop": record["suspected_repetition_loop"],
    })

review = pd.DataFrame(rows).sort_values(
    ["format_valid", "failure_penalty_cer"], ascending=[True, False],
)
display(review.round(4))

,sequence_id,format_valid,text_error_rate,failure_penalty_cer,speaker_accuracy,dialogue_f1,generated_tokens,hit_max_new_tokens,repetition_loop
1,seq_2032620aa4e4ac7f,False,NaN,1.0000,NaN,NaN,1024,True,False
3,seq_535e0754d805105b,False,NaN,1.0000,NaN,NaN,1024,True,False
6,seq_1c1d4f5e71a2f554,False,NaN,1.0000,NaN,NaN,1024,True,True
0,seq_d8bc3c6ac1bb6017,True,1.4645,1.4645,0.0000,0.1000,214,False,False
2,seq_d7af6c64ebc22bbe,True,1.1558,1.1558,0.6667,0.4615,245,False,False
7,seq_098e260f231aa789,True,0.9047,0.9047,0.6667,0.5357,606,False,False
4,seq_210721b0d576149f,True,0.8285,0.8285,0.7500,0.5333,207,False,False
5,seq_dce8be23c3281228,True,0.5672,0.5672,0.4286,0.5833,248,False,False


In [9]:
# Inspect the sequence with the worst result.
_ = inspect_prediction(
    EVAL_DIR / "predictions.jsonl",
    sequence_id=review.iloc[0]["sequence_id"],
    minimum_similarity=MINIMUM_SIMILARITY,
)

SEQUENCE
seq_2032620aa4e4ac7f

TARGET
[
  [
    {
      "speaker": "character1",
      "text": "HUH? DOES KOMABA SEEM DIFFERENT THAN USUAL?"
    },
    {
      "speaker": "character2",
      "text": "HE'S DOING REALLY WELL. IT LOOKS LIKE HE'S BEEN WORKING HARD FOR THE FALL TOURNAMENT."
    },
    {
      "speaker": "UNKNOWN",
      "text": "THAT'S THE WAY!"
    },
    {
      "speaker": "character1",
      "text": "NO, THAT'S NOT IT. IT'S LIKE HE'S..."
    },
    {
      "speaker": "character1",
      "text": "OUT FOR BLOOD."
    }
  ],
  [],
  [
    {
      "speaker": "UNKNOWN",
      "text": "WOAH!!"
    },
    {
      "speaker": "character3",
      "text": "OKAWA-SENPAI, YOU LOOK SO COOL!"
    },
    {
      "speaker": "character4",
      "text": "RIGHT HERE! THIS IS WHERE IT HAPPENS!"
    },
    {
      "speaker": "UNKNOWN",
      "text": "SO CLOSE!!"
    },
    {
      "speaker": "character5",
      "text": "DO YOU HAVE TO KEEP WATCHING THAT? IT'S EMBARRASSING."
    },
    {
     

## Official task score: primary experiment benchmark

Use the unmodified `TaskData/score.py` to judge text order and balanced text/speaker quality on the fixed holdout. Inspect invalid outputs and missing sequences before accepting an improvement.


In [ ]:
from Scripts.taskScore import score_saved_predictions

official_base = score_saved_predictions(BASELINE_DIR / "predictions.jsonl", LABELS_PATH, validation_ids, BASELINE_DIR / "task_score")
official_tuned = score_saved_predictions(EVAL_DIR / "predictions.jsonl", LABELS_PATH, validation_ids, EVAL_DIR / "task_score")
official_comparison = pd.DataFrame({"base_model": official_base["macro"], "tuned": official_tuned["macro"]})
official_comparison["delta"] = official_comparison["tuned"] - official_comparison["base_model"]
display(official_comparison.round(4))
print("Missing or invalid tuned sequences:", official_tuned["missing_sequences"])
